In [1]:
# streamAggCountry.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, sum as spark_sum
from pyspark.sql.types import StructType, StringType, IntegerType

In [2]:
# ==============================
# Configuration Kafka
# ==============================
KAFKA_TOPIC = "topicA"
KAFKA_BOOTSTRAP = "kafka:9092"

In [3]:
# ==============================
# Schéma des messages Kafka
# ==============================
schema = StructType() \
    .add("id", StringType()) \
    .add("first_name", StringType()) \
    .add("last_name", StringType()) \
    .add("country", StringType()) \
    .add("product_name", StringType()) \
    .add("gender", StringType()) \
    .add("quantity", StringType())

In [4]:
# ==============================
# SparkSession avec les JARs
# ==============================
spark = SparkSession.builder \
    .appName("StreamAggCountry") \
    .config(
        "spark.jars",
        "/opt/spark/jars/mongo-spark-connector_2.12-10.2.2.jar,"
        "/opt/spark/jars/spark-sql-kafka-0-10_2.12-3.5.0.jar"
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

25/12/17 22:20:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/17 22:20:40 WARN DependencyUtils: Local jar /opt/spark/jars/mongo-spark-connector_2.12-10.2.2.jar does not exist, skipping.
25/12/17 22:20:41 INFO SparkContext: Running Spark version 3.5.7
25/12/17 22:20:41 INFO SparkContext: OS info Linux, 6.6.87.2-microsoft-standard-WSL2, amd64
25/12/17 22:20:41 INFO SparkContext: Java version 11.0.29
25/12/17 22:20:41 INFO ResourceUtils: ==============================================================
25/12/17 22:20:41 INFO ResourceUtils: No custom resources configured for spark.driver.
25/12/17 22:20:41 INFO ResourceUtils: ==============================================================
25/12/17 22:20:41 INFO SparkContext: Submitted application: StreamAggCountry
25/12/17 22:20:41 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor

In [5]:
# ==============================
# Lecture du flux Kafka
# ==============================
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "latest") \
    .load()

Py4JJavaError: An error occurred while calling o32.load.
: java.lang.NoClassDefFoundError: org/apache/kafka/common/serialization/ByteArraySerializer
	at org.apache.spark.sql.kafka010.KafkaSourceProvider$.<init>(KafkaSourceProvider.scala:601)
	at org.apache.spark.sql.kafka010.KafkaSourceProvider$.<clinit>(KafkaSourceProvider.scala)
	at org.apache.spark.sql.kafka010.KafkaSourceProvider.org$apache$spark$sql$kafka010$KafkaSourceProvider$$validateStreamOptions(KafkaSourceProvider.scala:338)
	at org.apache.spark.sql.kafka010.KafkaSourceProvider.sourceSchema(KafkaSourceProvider.scala:71)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceSchema(DataSource.scala:233)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceInfo$lzycompute(DataSource.scala:118)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceInfo(DataSource.scala:118)
	at org.apache.spark.sql.execution.streaming.StreamingRelation$.apply(StreamingRelation.scala:36)
	at org.apache.spark.sql.str